In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, LassoCV
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt

In [6]:
# Model Comparison

This notebook mirrors the canonical script pipeline. All models use the shared feature list, preprocessing, and split utilities.

Shape: (242035, 17)


,COSMIC_ID,CELL_LINE_NAME,TCGA_DESC,DRUG_ID,DRUG_NAME,LN_IC50,GDSC Tissue descriptor 1,GDSC Tissue descriptor 2,Cancer Type (matching TCGA label),Microsatellite instability Status (MSI),Screen Medium,Growth Properties,CNA,Gene Expression,Methylation,TARGET,TARGET_PATHWAY
0,683667,PFSK-1,MB,1003,Camptothecin,-1.463887,nervous_system,medulloblastoma,MB,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
1,684057,ES5,UNCLASSIFIED,1003,Camptothecin,-3.360586,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
2,684059,ES7,UNCLASSIFIED,1003,Camptothecin,-5.044940,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
3,684062,EW-11,UNCLASSIFIED,1003,Camptothecin,-3.741991,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Adherent,Y,Y,Y,TOP1,DNA replication
4,684072,SK-ES-1,UNCLASSIFIED,1003,Camptothecin,-5.142961,bone,ewings_sarcoma,NaN,MSS/MSI-L,R,Semi-Adherent,Y,Y,Y,TOP1,DNA replication


In [ ]:
import numpy as np
import pandas as pd

from cancer_ml_utils import build_model_registry, load_model_dataframe, regression_metrics, split_dataset


In [ ]:
split_strategy = "random"  # change to "drug" or "cell_line" for stricter grouped evaluation
model_df = load_model_dataframe(include_group_columns=split_strategy != "random")
data_split = split_dataset(model_df, split_strategy=split_strategy)

print("Split strategy:", split_strategy)
print("Train:", data_split.X_train.shape)
print("Validation:", data_split.X_val.shape)
print("Test:", data_split.X_test.shape)


In [ ]:
results = []
trained_models = {}

for model_name, model in build_model_registry().items():
    model.fit(data_split.X_train, data_split.y_train)
    val_pred = model.predict(data_split.X_val)
    metrics = regression_metrics(data_split.y_val, val_pred)
    row = {"Model": model_name, **metrics}
    if model_name == "Lasso":
        row["Best Alpha"] = model.named_steps["model"].alpha_
        row["Nonzero Coefficients"] = int(np.count_nonzero(model.named_steps["model"].coef_))
    results.append(row)
    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(
    by=["RMSE", "MAE", "R2"],
    ascending=[True, True, False],
).reset_index(drop=True)

results_df


In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = trained_models[best_model_name]
y_test_pred = best_model.predict(data_split.X_test)
test_metrics = regression_metrics(data_split.y_test, y_test_pred)

print("Selected best model:", best_model_name)
print("Final Test Performance")
print("RMSE:", round(test_metrics["RMSE"], 4))
print("MAE :", round(test_metrics["MAE"], 4))
print("R2  :", round(test_metrics["R2"], 4))
